<center><img src='https://raw.githubusercontent.com/Jangrae/img/master/ml_python.png' width=600/></center>

<center><img src = "https://github.com/Jangrae/img/blob/master/airport2.png?raw=true" width=800 /></center>

# 실습 내용

- 여러 알고리즘으로 만든 모델의 성능을 교차 검증을 통해 예측해 봅니다.
- 성능이 좋을 것으로 예상되는 알고리즘으로 최적화된 성능의 모델을 만듭니다.
- 튜닝된 모델의 성능을 평가합니다.

# 1.환경 준비

- 기본 라이브러리와 대상 데이터를 가져와 이후 과정을 준비합니다.

In [ ]:
# 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

warnings.filterwarnings(action='ignore')
%config InlineBackend.figure_format = 'retina'

In [ ]:
# 데이터 불러오기
path = 'https://raw.githubusercontent.com/jangrae/csv/master/airline_satisfaction_small.csv'
data = pd.read_csv(path)

# 2.데이터 이해

- 분석할 데이터를 충분히 이해할 수 있도록 다양한 탐색 과정을 수행합니다.

**데이터 설명**

- id : 탑승자 고유 아이디
- gender: 성별 (Female, Male)
- customer_type: 고객 유형 (Loyal customer, disloyal customer)
- age: 탑승자 나이
- type_of_travel: 비행 목적(Personal Travel, Business Travel)
- class: 등급 (Business, Eco, Eco Plus)
- flight_distance: 비행 거리
- inflight_wifi_service: 와이파이 서비스 만족도 (0:N/A; 1-5)
- departure/arrival_time_convenient: 출발, 도착 시간 만족도 (0:N/A; 1-5)
- ease_of_online_booking: 온라인 부킹 만족도 (0:N/A; 1-5)
- gate_location: 게이트 위치 만족도 (0:N/A; 1-5)
- food_and_drink: 식사와 음료 만족도 (0:N/A; 1-5)
- online_boarding: 온라인 보딩 만족도 (0:N/A; 1-5)
- seat_comfort: 좌석 편안함 만족도 (0:N/A; 1-5)
- inflight_entertainment: 기내 엔터테인먼트 만족도 (0:N/A; 1-5)
- on-board_service: 온 보드 서비스 만족도 (0:N/A; 1-5)
- leg_room_service: 다리 공간 만족도 (0:N/A; 1-5)
- baggage_handling: 수하물 처리 만족도 (0:N/A; 1-5)
- check-in_service: 체크인 서비스 만족도 (0:N/A; 1-5)
- inflight_service: 기내 서비스 만족도 (0:N/A; 1-5)
- cleanliness: 청결 만족도 (0:N/A; 1-5)
- departure_delay_in_minutes: 출발 지연 시간(분)
- arrival_delay_in_minutes: 도착 지연 시간(분)
- **satisfaction: 항공 만족도(1: Satisfaction, 0: Neutral or Dissatisfaction) - Target**

In [ ]:
# 데이터 살펴보기
data.head()

In [ ]:
# 기술통계 확인
data.describe()

In [ ]:
# 변수 관련 정보 확인
data.info()

# 3.데이터 준비

- 전처리 과정을 통해 머신러닝 알고리즘에 사용할 수 있는 형태의 데이터를 준비합니다.

**1) 변수 제거**

In [ ]:
# 변수 제거: id, departure/arrival_time_convenient, gate_location, departure_delay_in_minutes
drop_cols = ['id', 'departure/arrival_time_convenient', 'gate_location', 'departure_delay_in_minutes']
data.drop(columns=drop_cols, inplace=True)

# 확인
data.head()

**2) 결측치 제거**

In [ ]:
# 결측치 제거
data.dropna(inplace=True)

# 확인
data.isnull().sum()

**3) x, y 분리**

In [ ]:
# x, y 분리
target = 'satisfaction'

x = data.drop(columns=target)
y = data.loc[:, target]

**4) 가변수화**

In [ ]:
# 가변수화
dumm_cols =['gender', 'customer_type', 'type_of_travel', 'class']
x = pd.get_dummies(x, columns=dumm_cols, drop_first=True, dtype=int)

# 확인
x.head()

**5) 학습용, 평가용 데이터 분리**

In [ ]:
# 학습용, 평가용 데이터 분리
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=1)

**6) 정규화**

In [ ]:
# 정규화
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(x_train)
x_train_s = scaler.transform(x_train)
x_test_s = scaler.transform(x_test)

# 4.모델링


- 하이퍼파라미터 최적화 과정을 통해 최선의 성능을 갖는 모델을 만들고 성능을 검증합니다.
- 우선 사용할 라이브러리를 모두 불러옵니다.

In [ ]:
# 불러오기
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report

**1) KNN**

- 최적으로 파라미터로 학습된 KNN 모델을 만들고, 최고의 파라미터와 성능을 확인합니다.
- 모델 이름은 model_knn으로 지정합니다.

In [ ]:
# 파라미터 선언
param = {'n_neighbors': range(2, 11)}

# 선언하기
model_knn = GridSearchCV(KNeighborsClassifier(),
                         param,
                         cv=5)

# 학습하기
model_knn.fit(x_train_s, y_train)

# 결과확인
print('* 파라미터:', model_knn.best_params_)
print('* 예측성능:', model_knn.best_score_)

**2) Decision Tree**

- 최적으로 파라미터로 학습된 Decision Tree 모델을 만들고, 최고의 파라미터와 성능을 확인합니다.
- 모델 이름은 model_dst로 지정합니다.

In [ ]:
# 파라미터 선언
param = {'max_depth': range(1, 21)}

# 선언하기
model_dst = GridSearchCV(DecisionTreeClassifier(), param, cv=5)

# 학습하기
model_dst.fit(x_train, y_train)

# 결과확인
print('* 파라미터:', model_dst.best_params_)
print('* 예측성능:', model_dst.best_score_)

**3) Random Forest**

- 최적으로 파라미터로 학습된 Random Forest 모델을 만들고, 최고의 파라미터와 성능을 확인합니다.
- 모델 이름은 model_rdf로 지정합니다.

In [ ]:
# 파라미터 선언
param = {'max_depth': range(1, 21)}

# 선언하기
model_rdf = GridSearchCV(RandomForestClassifier(), param, cv=5)

# 학습하기
model_rdf.fit(x_train, y_train)

# 결과확인
print('* 파라미터:', model_rdf.best_params_)
print('* 예측성능:', model_rdf.best_score_)

# 5.성능평가

- 가장 좋은 성능을 보일 것으로 예상되는 모델로 평가를 진행합니다.

In [ ]:
# 예측하기
y_pred = model_rdf.predict(x_test)

# 평가하기
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))